# SEISMICPIPELINE: PREDICCION DE RIESGO DE TSUNAMI MEDIANTE XGBOOST

<br>

**Institucion:** Universidad Internacional del Ecuador (UIDE)

**Escuela:** Ciencias de la Computacion

**Asignaturas:** Big Data - Machine Learning - Gestion de Proyectos de SI - Ciberseguridad

**Semestre:** Sexto - 2026

**Jira:** Proyecto SEIS

**Repositorio:** github.com/DanielSozoranga/tsunami-risk-pipeline

<br>

**Equipo Scrum:**

| Integrante | Roles |
|---|---|
| **Daniel Sozoranga** | Scrum Master · Development Team |
| **Ricardo Álvarez** | Product Owner · Development Team |

<br>

---

## PREGUNTA QUE RESPONDE EL MODELO

**?Este sismo generara tsunami?**

El modelo es un clasificador binario **XGBoost** que, dadas las caracteristicas fisicas de un sismo, devuelve una **probabilidad continua entre 0 y 1** (`predict_proba`) de que el evento genere tsunami. Esa probabilidad se proyecta sobre el registro sismico costero ecuatoriano como un **score de riesgo por provincia** (Esmeraldas, Manabi, Santa Elena, Guayas, El Oro, Galapagos), no como clasificacion binaria.

<br>

---

## ESTRUCTURA DEL NOTEBOOK

| Seccion | Contenido |
|---|---|
| **Seccion 1** | Instalacion y Configuracion del Entorno |
| **Seccion 2** | Importacion de Librerias |
| **Seccion 3** | Conexion y Extraccion de Datos (API USGS) |
| **Seccion 4** | Variables Obtenidas - Dataset Original |
| **Seccion 5** | Renombrado de Variables: API -> Espanol |
| **Seccion 6** | Limpieza y Preprocesamiento (ETL) |
| **Seccion 7** | Analisis Exploratorio (EDA) |
| **Seccion 8** | Benchmarking de Engines: Pandas vs PySpark vs Dask |
| **Seccion 9** | Preparacion del Dataset de Machine Learning |
| **Seccion 10** | Entrenamiento del Clasificador XGBoost |
| **Seccion 11** | Diagnostico de Entrenamiento |
| **Seccion 12** | Evaluacion del Modelo |
| **Seccion 13** | Modelos Baseline de Comparacion |
| **Seccion 14** | Justificacion Tecnica de la Seleccion de XGBoost |
| **Seccion 15** | Proyeccion de Riesgo sobre Ecuador |
| **Seccion 16** | Validacion Internacional (Tohoku 2011 / Maule 2010) |
| **Seccion 17** | Dashboard Interactivo de Resultados |
| **Seccion 18** | Analisis Critico: Limitaciones y Mejoras |
| **Seccion 19** | Conclusiones |

<br>

---

## CONVENCIONES

- **Semilla aleatoria global:** `SEED = 42` - reutilizada en todo split y modelo para garantizar reproducibilidad.
- **Idioma de las variables:** los datos llegan de la API con nombres en ingles. En la Seccion 5 se renombran a espanol con tabla de equivalencias original -> final.
- **Features del modelo (7):** `magnitud`, `profundidad_km`, `latitud`, `longitud`, `significancia`, `num_estaciones`, `brecha_azimutal`. La seleccion se justifica en la Seccion 7 con la matriz de correlacion completa.
- **Target:** `tsunami` (0 = no genero tsunami, 1 = si genero tsunami), bandera oficial del catalogo USGS.
- **Modelo unico:** XGBoost. Random Forest y Regresion Logistica aparecen solo como baselines de comparacion (Seccion 13).

<br>

---

# SECCION 1 - INSTALACION Y CONFIGURACION DEL ENTORNO

<br>

Se instalan las librerias que no estan disponibles por defecto en Google Colab.

| Libreria | Proposito |
|---|---|
| **xgboost** | Algoritmo principal del proyecto (clasificador de tsunamis) |
| **pyspark** | Procesamiento distribuido (motor Big Data del benchmarking) |
| **dask** | Procesamiento paralelo con ejecucion lazy (benchmarking) |
| **plotly** | Dashboard interactivo de resultados |
| **psutil** | Medicion de consumo de memoria RAM en tiempo real |
| **joblib** | Serializacion del modelo entrenado |

<br>

In [ ]:
# Librerias que Google Colab no tiene preinstaladas.
# Si ejecutas localmente, asegurate de tener Python 3.10+
# y un entorno virtual activo antes de correr este comando.
!pip install xgboost pyspark "dask[dataframe]" plotly psutil joblib --quiet

print("Librerias instaladas correctamente")


---

# SECCION 2 - IMPORTACION DE LIBRERIAS

<br>

In [ ]:
# --- Consumo de API y utilidades ---
import os
import time
import math
import json
import warnings
import requests

# --- Manipulacion de datos ---
import numpy  as np
import pandas as pd

# --- Visualizacion estatica ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Visualizacion interactiva ---
import plotly.graph_objects as go
import plotly.express       as px
from   plotly.subplots      import make_subplots

# --- Machine Learning ---
from sklearn.model_selection  import (train_test_split, StratifiedKFold,
                                      cross_val_score)
from sklearn.metrics          import (roc_auc_score, roc_curve,
                                      confusion_matrix, classification_report,
                                      precision_score, recall_score, f1_score,
                                      ConfusionMatrixDisplay)
from sklearn.ensemble         import RandomForestClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import StandardScaler
from xgboost                  import XGBClassifier
import xgboost as xgb

# --- Persistencia y metricas del sistema ---
import joblib   # Serializacion del modelo entrenado
import psutil   # Medicion de consumo de RAM en tiempo real

# --- Configuracion global de visualizacion ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)

# Semilla global: garantiza reproducibilidad en splits, modelos y muestras
SEED = 42
np.random.seed(SEED)

print("Librerias importadas correctamente")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  xgboost : {xgb.__version__}")
print(f"  SEED    : {SEED}")


---

# SECCION 3 - CONEXION Y EXTRACCION DE DATOS (API USGS)

<br>

## 3.1 Descripcion de la Fuente

<br>

| Parametro | Detalle |
|---|---|
| **Endpoint** | https://earthquake.usgs.gov/fdsnws/event/1/query |
| **Autenticacion** | Ninguna (API 100% publica) |
| **Estandar** | FDSN Event Web Service Specification |
| **Formato de respuesta** | GeoJSON (FeatureCollection) |
| **Cobertura temporal** | 1900 - presente |
| **Cobertura geografica** | Global |

<br>

## 3.2 Parametros de Consulta

<br>

Se extrae el **Cinturon de Fuego del Pacifico (1990-2024)** con los siguientes parametros:

| Parametro | Valor | Justificacion |
|---|---|---|
| `minmagnitude` | 5.0 | Eventos con potencial tsunamigenico real |
| `minlatitude` / `maxlatitude` | -60 / 65 | Desde la placa Antartica hasta Alaska/Kamchatka |
| `minlongitude` / `maxlongitude` | 110 / 300 | Cuenca del Pacifico (300 = -60W, permite cruzar el antimeridiano) |
| `limit` | 20000 | Limite maximo de la API por request -> requiere paginacion anual |

La paginacion anual (1990-2024 = 35 requests) evita superar el limite de la API. El retry con backoff exponencial protege contra fallas transitorias de la red.

<br>

## 3.3 Configuracion del Entorno de Archivos

<br>

In [ ]:
# =============================================================
#  SECCION 3.3 - CONFIGURACION DE CARPETAS
# =============================================================
#  Estructura estandar del proyecto:
#    /data/raw        <- datasets crudos directamente de la API
#    /data/processed  <- datasets limpios y artefactos del modelo
#    /notebooks       <- notebooks de analisis y exploracion
#    /models          <- modelos serializados (.pkl)
# =============================================================

try:
    # Entorno Google Colab: monta Drive automaticamente
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/SeismicPipeline'
except ModuleNotFoundError:
    # Entorno local: crea las carpetas relativas al directorio actual
    BASE = os.path.abspath('./SeismicPipeline')

DIRS = {
    'raw'       : f'{BASE}/data/raw',
    'processed' : f'{BASE}/data/processed',
    'notebooks' : f'{BASE}/notebooks',
    'models'    : f'{BASE}/models',
}

for nombre, ruta in DIRS.items():
    os.makedirs(ruta, exist_ok=True)   # exist_ok: no falla si ya existe
    print(f"  [OK] {nombre:10s} -> {ruta}")

print()
print("Estructura de carpetas configurada correctamente")


<br>

## 3.4 Funciones de Extraccion

<br>

In [ ]:
# =============================================================
#  SECCION 3.4 - FUNCIONES DE EXTRACCION
# =============================================================

# URL base del servicio FDSN (Federal Digital Seismograph Network)
BASE_URL = 'https://earthquake.usgs.gov/fdsnws/event/1/query'

# Bounding box del Cinturon de Fuego del Pacifico.
# La API acepta longitudes > 180 para cruzar el antimeridiano
# (ej. 300 = 360 - 60 = 60 W), lo que permite cubrir el Pacifico
# sin necesidad de dividir la consulta.
BBOX_RING_OF_FIRE = {
    'minlatitude'  : -60,   # Desde la placa Antartica
    'maxlatitude'  :  65,   # Hasta Alaska / Kamchatka
    'minlongitude' : 110,   # Extremo occidental: Filipinas
    'maxlongitude' : 300,   # Extremo oriental: Costa Rica (via antimeridiano)
}

MIN_MAGNITUD = 5.0   # Umbral minimo: eventos con potencial tsunamigenico real
ANIO_INICIO  = 1990
ANIO_FIN     = 2024


def consultar_usgs(params: dict, timeout: int = 60) -> dict:
    '''
    Realiza un GET a la API USGS y retorna el JSON decodificado.

    Parametros:
        params  : dict - parametros de la consulta HTTP
        timeout : int  - segundos maximos de espera (default 60)

    Retorna:
        dict con la respuesta GeoJSON de la API.

    Lanza:
        RuntimeError si la respuesta es un error HTTP, timeout o falla de red.
    '''
    try:
        resp = requests.get(BASE_URL, params=params, timeout=timeout)

        # Errores del cliente (400-499): problema en los parametros enviados
        if 400 <= resp.status_code < 500:
            raise RuntimeError(f"Error de cliente {resp.status_code}: {resp.text[:200]}")

        # Errores del servidor (500+): USGS no disponible temporalmente
        if resp.status_code >= 500:
            raise RuntimeError(f"Error de servidor {resp.status_code}: USGS no disponible")

        data = resp.json()

        # La API USGS siempre incluye 'features' en una respuesta valida
        if 'features' not in data:
            raise RuntimeError(f"Respuesta inesperada: {str(data)[:200]}")

        return data

    except requests.exceptions.Timeout:
        raise RuntimeError(f"Timeout: la API no respondio en {timeout}s")
    except requests.exceptions.ConnectionError as e:
        raise RuntimeError(f"Error de conexion: {e}")


def consultar_con_retry(params: dict, max_reintentos: int = 4) -> dict:
    '''
    Envuelve consultar_usgs() con reintentos de backoff exponencial.

    Parametros:
        params         : dict - parametros de la consulta
        max_reintentos : int  - numero maximo de reintentos (default 4)

    Estrategia de espera: 2^(intento+1) segundos -> 2, 4, 8, 16 seg.
    Esto evita saturar la API cuando hay fallas transitorias de red.
    '''
    for intento in range(max_reintentos + 1):
        try:
            return consultar_usgs(params)
        except RuntimeError as e:
            if intento == max_reintentos:
                raise   # Agotados los reintentos: propagar el error
            espera = 2 ** (intento + 1)
            print(f"  [retry {intento+1}/{max_reintentos}] {e} - esperando {espera}s...")
            time.sleep(espera)


# Prueba de conexion con 3 eventos recientes para verificar que la API responde
prueba   = consultar_usgs({'format': 'geojson', 'minmagnitude': 6, 'limit': 3, 'orderby': 'time'})
n_prueba = prueba.get('metadata', {}).get('count', len(prueba['features']))
print(f"Conexion a la API USGS: OK")
print(f"  Eventos recibidos en prueba: {n_prueba}")


<br>

## 3.5 Extraccion Masiva Ring of Fire 1990-2024

<br>

La extraccion trae **todas las propiedades disponibles de la API** con sus nombres originales en ingles. El renombrado a espanol se realiza en la Seccion 5, de forma trazable, para que el docente pueda comparar el dataset original con el dataset transformado.

<br>

In [ ]:
# =============================================================
#  SECCION 3.5 - EXTRACCION MASIVA CON PAGINACION ANUAL
# =============================================================
#  La API limita cada request a 20,000 eventos.
#  Para cubrir 1990-2024 (35 anios) se hacen 35 requests,
#  uno por anio. Total esperado: ~15,000 - 30,000 eventos.
# =============================================================

# Propiedades extraidas de la API - se conservan los NOMBRES ORIGINALES
# (el renombrado a espanol ocurre en la Seccion 5)
PROPS_API = [
    'mag',     # Magnitud del sismo
    'sig',     # Indice de significancia USGS
    'nst',     # Numero de estaciones que reportaron
    'gap',     # Brecha azimutal (calidad de localizacion)
    'dmin',    # Distancia a la estacion mas cercana
    'rms',     # Error cuadratico medio del ajuste de tiempo
    'felt',    # Numero de reportes de personas que lo sintieron
    'cdi',     # Intensidad maxima comunitaria (CDI)
    'mmi',     # Intensidad maxima instrumental (MMI)
    'alert',   # Nivel de alerta PAGER
    'magType', # Tipo de magnitud (Mw, mb, ml, etc.)
    'tsunami', # Bandera oficial: 1 si el evento genero tsunami
    'place',   # Descripcion textual del lugar
    'time',    # Timestamp en milisegundos UTC
]


def extraer_ring_of_fire() -> pd.DataFrame:
    '''
    Extrae el catalogo sismico del Cinturon de Fuego del Pacifico (1990-2024).

    Realiza 35 requests anuales a la API USGS paginando por anio para respetar
    el limite de 20,000 eventos por consulta. Incluye un log de extraccion por anio.

    Retorna:
        pd.DataFrame con todos los eventos y sus propiedades originales de la API.
    '''
    registros = []   # Acumula cada evento como diccionario
    log       = []   # Registro de cuantos eventos se obtuvieron por anio

    for anio in range(ANIO_INICIO, ANIO_FIN + 1):
        params = {
            'format'       : 'geojson',
            'starttime'    : f'{anio}-01-01',
            'endtime'      : f'{anio}-12-31T23:59:59',
            'minmagnitude' : MIN_MAGNITUD,
            'limit'        : 20000,          # Maximo permitido por la API
            **BBOX_RING_OF_FIRE,
        }
        data  = consultar_con_retry(params)
        feats = data.get('features', [])

        for f in feats:
            p, g = f['properties'], f['geometry']['coordinates']
            # La geometria GeoJSON devuelve [longitud, latitud, profundidad]
            fila = {
                'id'        : f['id'],
                'longitude' : g[0],   # Longitud del epicentro
                'latitude'  : g[1],   # Latitud del epicentro
                'depth'     : g[2],   # Profundidad del hipocentro en km
            }
            for prop in PROPS_API:
                fila[prop] = p.get(prop)   # None si la API no reporta la propiedad
            registros.append(fila)

        log.append({'anio': anio, 'eventos': len(feats)})
        print(f"  {anio}: {len(feats):5,d} eventos extraidos")
        time.sleep(0.5)   # Pausa de cortesia para no saturar la API (rate limiting)

    df = pd.DataFrame(registros)
    pd.DataFrame(log).to_csv(f"{DIRS['raw']}/log_extraccion.csv", index=False)
    return df


# Estrategia de carga: reutilizar el dataset si ya existe en Drive.
# Si la version guardada no tiene todas las columnas (version antigua),
# se re-extrae automaticamente para garantizar integridad.
RUTA_RAW = f"{DIRS['raw']}/usgs_raw.csv"

if os.path.exists(RUTA_RAW):
    df_raw = pd.read_csv(RUTA_RAW)
    if 'dmin' not in df_raw.columns:
        print("Version anterior detectada (columnas incompletas). Re-extrayendo...")
        df_raw = extraer_ring_of_fire()
        df_raw.to_csv(RUTA_RAW, index=False)
    else:
        print("Dataset crudo encontrado en Drive. Cargando sin re-extraer...")
else:
    print("Primera ejecucion: iniciando extraccion masiva 1990-2024 (~2 minutos)...")
    df_raw = extraer_ring_of_fire()
    df_raw.to_csv(RUTA_RAW, index=False)

print()
print("=" * 58)
print("  RESUMEN DE EXTRACCION - API USGS Earthquake Catalog")
print("=" * 58)
print(f"  Filas       : {df_raw.shape[0]:>10,d} registros")
print(f"  Columnas    : {df_raw.shape[1]:>10,d} variables")
print(f"  Periodo     :       1990 - 2024 (Ring of Fire)")
print(f"  Magnitud    :       M >= {MIN_MAGNITUD}")
print(f"  Exportado a : {RUTA_RAW}")
print("=" * 58)


---

# SECCION 4 - VARIABLES OBTENIDAS - DATASET ORIGINAL

<br>

Esta seccion muestra el dataset **tal como llega de la API USGS**, con sus nombres originales en ingles, antes de cualquier transformacion. Esto permite comparar el dataset original con el dataset transformado en la Seccion 5.

<br>

## 4.1 Catalogo de Variables de la API

<br>

Todas las propiedades que el endpoint GeoJSON de la USGS puede entregar para cada evento sismico:

| Variable (API) | Tipo | Descripcion |
|---|---|---|
| `id` | str | Identificador unico del evento |
| `longitude` | float | Longitud del epicentro (geometry[0]) |
| `latitude` | float | Latitud del epicentro (geometry[1]) |
| `depth` | float | Profundidad del hipocentro en km (geometry[2]) |
| `mag` | float | Magnitud del sismo |
| `magType` | str | Tipo de escala de magnitud (mww, mb, ml...) |
| `sig` | int | Indice de significancia USGS (0-1000+) |
| `nst` | int | Numero de estaciones sismicas utilizadas |
| `gap` | float | Brecha azimutal maxima en grados |
| `dmin` | float | Distancia a la estacion mas cercana (grados) |
| `rms` | float | Error RMS del ajuste temporal de la solucion |
| `felt` | float | Numero de reportes publicos (Did You Feel It?) |
| `cdi` | float | Intensidad maxima percibida por la comunidad |
| `mmi` | float | Intensidad instrumental maxima (ShakeMap) |
| `alert` | str | Nivel de alerta PAGER (green/yellow/orange/red) |
| `tsunami` | int | **TARGET**: indica si el evento genero tsunami (0/1) |
| `place` | str | Descripcion textual de la ubicacion |
| `time` | int | Timestamp Unix del evento (milisegundos) |

<br>

## 4.2 Vista del Dataset Original

<br>

In [ ]:
# =============================================================
#  SECCION 4.2 - VISTA DEL DATASET ORIGINAL (nombres API)
# =============================================================

print("Dataset original - primeras 5 filas (nombres originales de la API USGS):")
print()
display(df_raw.head())

print()
print("Columnas originales de la API:")
print(list(df_raw.columns))

<br>

## 4.3 Estructura y Tipos de Datos

<br>

In [ ]:
# =============================================================
#  SECCION 4.3 - ESTRUCTURA DEL DATASET ORIGINAL
# =============================================================

print("Estructura del dataset crudo:")
print()
df_raw.info()

print()
print("=" * 58)
print("  RESUMEN DE VALORES NULOS - Dataset Original")
print("=" * 58)
nulos = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
nulos = nulos[nulos > 0].sort_values(ascending=False)
for var, pct in nulos.items():
    n = df_raw[var].isnull().sum()
    print(f"  {var:12s} : {n:6,d} nulos  ({pct:5.1f}%)")
print("=" * 58)

---

> **Nota de desarrollo:** Esta seccion corresponde al issue **SEIS-8** (extraccion masiva Ring of Fire 1990-2024).
> Cubre:
> - SEIS-43: bbox geografico del Cinturon de Fuego con coordenadas exactas
> - SEIS-44: loop de paginacion anual 1990-2024 (35 requests)
> - SEIS-45: retry con backoff exponencial para rate limiting
> - SEIS-46: validacion del dataset crudo + exportacion a `/data/raw/usgs_raw.csv`
> 
> El ETL, EDA y modelado continuan en commits siguientes.
